# 02 — Entrenar VolleyDetector ③ (custom, sin librería YOLO)

Entrena nuestra red `VolleyDetector` (definida en `src/models/detector.py`) con el dataset
`voley.yolov8`. T4 free de Colab tarda ~30–45 min con 80 épocas.

Output: `weights/detector.pt` (descarga al final manualmente al PC para la GUI).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf volley-ai
!git clone https://github.com/Angelote567/DeepLearning.git volley-ai
%cd volley-ai
!pip install -q torch torchvision tqdm Pillow PyYAML
import os
os.makedirs('data', exist_ok=True)
os.makedirs('weights', exist_ok=True)
# Symlinks a Drive (donde ya está el dataset descargado por notebook 01)
if not os.path.exists('data/voley.yolov8'):
    os.symlink('/content/drive/MyDrive/volley-ai/data/voley.yolov8', 'data/voley.yolov8')
if not os.path.islink('weights') and not os.listdir('weights'):
    os.rmdir('weights')
    os.symlink('/content/drive/MyDrive/volley-ai/weights', 'weights')

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
from src.train.train_detector import train
best = train(epochs=80, batch_size=16, lr=1e-3)
print('Mejores pesos:', best)

In [ ]:
# Evaluación rápida con un par de imágenes del test set
import torch, cv2
from PIL import Image
import torchvision.transforms.functional as TF
from src.models.detector import VolleyDetector
from src.config import IMG_SIZE, CLASSES

model = VolleyDetector().cuda().eval()
ckpt = torch.load('weights/detector.pt', map_location='cuda')
model.load_state_dict(ckpt['model'])

import glob, random
for path in random.sample(glob.glob('data/voley.yolov8/test/images/*.jpg'), 4):
    img = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    x = TF.to_tensor(img).unsqueeze(0).cuda()
    out = model.predict(x)[0]
    print(path, len(out['boxes']), 'detecciones')